# Etapa 3 — CNNs, ViTs e Transfer Learning (Colab)
Notebook para rodar a Etapa 3 com GPU.

**Antes de começar:** `Ambiente de execução > Alterar tipo de hardware > GPU (T4)`.

Fluxo: GPU → clonar repo → instalar deps → (opcional) cachear dataset no Drive → montar loaders 224×224 → varredura otimizador×LR → comparação de modelos.

## 1. Conferir a GPU

In [ ]:
!nvidia-smi

## 2. Clonar o repositório
Troque pela URL do repositório de vocês.

In [ ]:
REPO_URL = 'https://github.com/SEU_USUARIO/trabalho-final-ia.git'  # <-- EDITE
!git clone $REPO_URL
%cd trabalho-final-ia

## 3. Instalar dependências

In [ ]:
!pip install -q -r requirements.txt

## 4. (Opcional) Cachear o dataset no Google Drive
O PathMNIST 224×224 é grande (~3 GB). Salvar no Drive evita rebaixar a cada sessão.
Pule esta célula se preferir baixar para a sessão (some ao desconectar).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/pathmnist'  # ou None para baixar na sessão
import os; os.makedirs(DATA_ROOT, exist_ok=True)

## 5. Montar os DataLoaders 224×224
A 1ª execução baixa o dataset (demora). Ajuste `batch_size` se a VRAM estourar.

In [ ]:
from src.utils.seeds import set_seed
from src.etapa2_pytorch.data_224 import make_loaders
set_seed(42)
DATA_ROOT = globals().get('DATA_ROOT', None)
train_loader, val_loader, test_loader = make_loaders(
    size=224, batch_size=64, num_workers=2, augment=True, root=DATA_ROOT)
print('batches treino:', len(train_loader), '| val:', len(val_loader))

## 6. Varredura: 2 otimizadores × 3 learning rates
Roda no ResNet50 (fine-tuning) como modelo representativo para escolher a melhor config.
Salva `results/logs/etapa3_sweep_optim_lr.csv`.

In [ ]:
from src.etapa3_cnn_vit.run_experiments import sweep_optim_lr
melhor = sweep_optim_lr('resnet50', 'fine_tuning',
                        train_loader, val_loader, epochs=5, pretrained=True)
print('Melhor config:', melhor)

## 7. Comparação de todos os modelos
CNN autoral + 3 CNNs (feature extraction e fine-tuning) + ViT, com a melhor config.
Salva `results/logs/etapa3_comparacao_modelos.csv` e os checkpoints do melhor de cada um.

In [ ]:
from src.etapa3_cnn_vit.run_experiments import compare_models
resultados = compare_models(train_loader, val_loader,
                            optimizer=melhor['otimizador'], lr=melhor['lr'],
                            epochs=8, pretrained=True)
import pandas as pd
pd.DataFrame(resultados)

## 8. (Mais tarde) Avaliação no conjunto de TESTE
⚠️ Regra do enunciado: o teste só pode ser usado **uma única vez**, sobre o **melhor modelo final** (provavelmente após a Etapa 5). Não rode isto durante a exploração.

In [ ]:
# from src.etapa3_cnn_vit.engine import evaluate, build... (carregar o melhor checkpoint)
# acc_teste = evaluate(modelo_final, test_loader)
# print('Acuracia no TESTE (uso unico):', acc_teste)